In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Local data folder. Set this in your shell before launching Jupyter or VS Code.
data_dir = os.environ.get('RETAIL_DATA_DIR')
if not data_dir:
    raise EnvironmentError('Set RETAIL_DATA_DIR to your local Retail Business data folder before running this notebook.')
DATA_DIR = Path(data_dir).expanduser()

# Function to find the latest files within the folders: returns latest file
def latest_file(folder, suffix):
    matches = sorted(folder.glob(f"*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"No files matching *{suffix} in {folder}")
    return matches[-1]


# Each table lives in it's own folder:
folders={
    'customers': DATA_DIR / 'Dim_Customers',
    'orders': DATA_DIR / 'Fact_Orders',
    'products': DATA_DIR / 'Dim_Products',
    'line items': DATA_DIR / 'Dim_LineItems',
}

suffixes = {
    'customers': '_Customers.csv',
    'orders': '_Orders.csv',
    'products': '_Products.csv',
    'line items': '_Line Items.csv',
}

# Empty string to hold table names 
tables = {}

# Quick exploration of each table
for name, path in folders.items():
    df = pd.read_csv(latest_file(path, suffixes[name]))
    tables[name] = df
    print(f'=== {name.upper()} ===')
    print(df.shape)
    print(df.columns.tolist())
    df.info()

In [ ]:
# create dictionary for iteration of tables for grain verification
keys = {
    'customers': 'ID',
    'orders': 'ID',
    'products': 'ID', 
    'line items': 'ID'
}

# check the grain of the rows: what does 1 row represent? 
for name, col in keys.items():
    print(f"{name}:", tables[name][col].nunique(), "unique vs", len(tables[name]), "rows")


Hypotheses about the row counts:
    1. 

# Customer Table Exploration

In [ ]:
customers = tables['customers']

In [ ]:
# BEFORE cleaning duplicates
print("Rows before: ", len(customers))
print("Unique rows: ", customers['ID'].nunique())
print("Most repeated ID: ", customers['ID'].value_counts().max(), "times")

# Remove duplicates
customers_drop_dup = customers.drop_duplicates('ID', keep='last')

# AFTER cleaning duplicates
print("Rows after: ", len(customers_drop_dup))
print("Unique rows: ", customers_drop_dup['ID'].nunique())
print("Most repeated ID: ", customers_drop_dup['ID'].value_counts().max(), "times")


In [ ]:
top_id = customers['ID'].value_counts().idxmax()   # the most-repeated customer
customers[customers['ID'] == top_id]

# Order Table Exploration

In [ ]:
# orders table exploration, cleaning of duplicates
tables['orders'].head(10)

orders = tables['orders'].drop_duplicates('Name')

orders.head(10)

total_orders = orders['Name'].nunique() # should equal total rows
total_revenue = orders.drop_duplicates(subset='Name')['Shop total price'].sum()
total_customer_rows = len(orders['Name']) # should equal total orders

print(total_revenue)
print(f"{total_orders}", "unique orders vs", total_customer_rows, "rows")

- validated that orders table grain was showing one row per line item instead of one row per order
- removing duplicate 'Name' column items matched the grain to unique rows

In [ ]:
sum_of_nulls_orders = orders.isna().sum()
print("===== Columns with NULL values =====")
print(sum_of_nulls_orders)

missing_pct = orders.isna().mean().sort_values(ascending=False)
print("===== Missing Values Percentage per column =====")
print(missing_pct)

rows_fully_duplicated = orders.duplicated().sum()
print("===== Rows that are fully duplicated =====")
print(rows_fully_duplicated)

unique_values_per_column = orders.nunique()
print("===== Unique Values per Column =====")
print(unique_values_per_column)

In [ ]:
tables['customers'].isna().sum()

In [ ]:
print(tables['customers'].columns)
customers = tables['customers'].drop_duplicates('ID')
customers['Full Name'] = customers['First name'] + " " + customers['Last name']
top_15_customers_total_spend = customers.groupby(by='Full Name')['Amount spent'].sum().sort_values(ascending=False).head(15).reset_index()
# is the group by method necessary for this? 
print(top_15_customers_total_spend)
sns.barplot(data=top_15_customers_total_spend, x='Full Name', y='Amount spent')